In [16]:
!pip install transformers sentencepiece sumy rouge-score beautifulsoup4 requests -q


# TAHAP SCRAPPING DATA MENTAH

In [17]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

urls = [
    "https://albumkisahwayang.blogspot.com/2018/01/gandawardaya.html",
    "https://albumkisahwayang.blogspot.com/2018/01/bambang-danasalira.html"
]

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36"
}

data = []

def clean_text(text):
    text = re.sub(r"Baca juga:.*", "", text)
    text = re.sub(r"Untuk daftar.*", "", text)
    text = re.sub(r"Klik di sini.*", "", text)
    text = re.sub(r"Kediri,.*", "", text)
    text = re.sub(r"------------------------------", " ", text)
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

for url in urls:
    print(f"Scraping: {url}")
    try:
        res = requests.get(url, headers=headers, timeout=10)
        res.raise_for_status()
        soup = BeautifulSoup(res.text, "html.parser")

        article = soup.find("div", {"class": "post-body"})
        if article:
            raw_text = article.get_text(separator=" ", strip=True)
            clean = clean_text(raw_text)
            data.append({
                "url": url,
                "text_part": clean
            })
            print(f"Berhasil ambil {len(clean)} karakter dari {url}")
        else:
            print(f"Tidak ditemukan {url}")

        time.sleep(2)
    except Exception as e:
        print(f"Gagal: {url} -> {e}")

df = pd.DataFrame(data)
df.to_csv("wayang_dataset.csv", index=False, encoding="utf-8-sig")
print("\nDataset dibuat: wayang_dataset.csv")
df.head()


Scraping: https://albumkisahwayang.blogspot.com/2018/01/gandawardaya.html
Berhasil ambil 21296 karakter dari https://albumkisahwayang.blogspot.com/2018/01/gandawardaya.html
Scraping: https://albumkisahwayang.blogspot.com/2018/01/bambang-danasalira.html
Berhasil ambil 25030 karakter dari https://albumkisahwayang.blogspot.com/2018/01/bambang-danasalira.html

Dataset dibuat: wayang_dataset.csv


,url,text_part
0,https://albumkisahwayang.blogspot.com/2018/01/...,Kisah ini menceritakan kemunculan Raden Gandaw...
1,https://albumkisahwayang.blogspot.com/2018/01/...,Kisah ini menceritakan tentang perkawinan anta...


# Summarization (2 metode: Extractive & Abstractive)

In [22]:
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lsa import LsaSummarizer
from transformers import pipeline

abstractive = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6", framework="pt")

def extractive_summary(text, sentences=3):
    parser = PlaintextParser.from_string(text, Tokenizer("english"))
    summarizer = LsaSummarizer()
    summary = summarizer(parser.document, sentences)
    return " ".join(str(sentence) for sentence in summary)

def abstractive_summary_long(text, chunk_size=300):
    """
    Pecah teks panjang menjadi beberapa bagian agar tidak melebihi batas token.
    """
    text_chunks = []
    words = text.split()
    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i + chunk_size])
        text_chunks.append(chunk)

    summaries = []
    for chunk in text_chunks:
        try:
            result = abstractive(chunk, max_length=150, min_length=50, do_sample=False)
            summaries.append(result[0]["summary_text"])
        except Exception as e:
            print("Gagal ringkas chunk:", e)
            continue
    return " ".join(summaries)

extractive_results = []
abstractive_results = []

for i, t in enumerate(df["text_part"]):
    print(f"Proses teks {i+1}/{len(df)}...")
    ext_sum = extractive_summary(t, sentences=3)
    abs_sum = abstractive_summary_long(t)
    extractive_results.append(ext_sum)
    abstractive_results.append(abs_sum)

df["extractive_summary"] = extractive_results
df["abstractive_summary"] = abstractive_results
df.to_csv("wayang_summarization_results.csv", index=False, encoding="utf-8-sig")

print("Ringkasan selesai dibuat!")
df.head()


Device set to use cpu


Proses teks 1/2...
Proses teks 2/2...
Ringkasan selesai dibuat!


,url,text_part,extractive_summary,abstractive_summary
0,https://albumkisahwayang.blogspot.com/2018/01/...,Kisah ini menceritakan kemunculan Raden Gandaw...,Bambang Gandawardaya menjawab dirinya sejak ma...,Kisah ini saya olah dan saya kembangkan dari ...
1,https://albumkisahwayang.blogspot.com/2018/01/...,Kisah ini menceritakan tentang perkawinan anta...,"Akan tetapi, ia berkhayal alangkah indahnya ji...",Kisah ini menceritakan tentang perkawinan ant...


In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
results = []

for i, row in df.iterrows():
    ref = row["text_part"][:2000]  
    ext = row["extractive_summary"]
    abs_sum = row["abstractive_summary"]

    score_ext = scorer.score(ref, ext)
    score_abs = scorer.score(ref, abs_sum)

    results.append({
        "url": row["url"],
        "rouge1_extractive": score_ext["rouge1"].fmeasure,
        "rouge1_abstractive": score_abs["rouge1"].fmeasure,
        "rougeL_extractive": score_ext["rougeL"].fmeasure,
        "rougeL_abstractive": score_abs["rougeL"].fmeasure
    })

eval_df = pd.DataFrame(results)
eval_df


,url,rouge1_extractive,rouge1_abstractive,rougeL_extractive,rougeL_abstractive
0,https://albumkisahwayang.blogspot.com/2018/01/...,0.301449,0.458259,0.185507,0.227353
1,https://albumkisahwayang.blogspot.com/2018/01/...,0.139130,0.341463,0.075362,0.182114


Model	ROUGE-1	ROUGE-L	Interpretasi
Extractive	0.30 → 0.14	0.18 → 0.07	Cukup rendah — hanya mengambil kalimat dari teks asli.
Abstractive	0.46 → 0.34	0.22 → 0.18	Lebih tinggi — model memahami makna dan menyusun ulang kalimat.